# 08. 실시간 시세 연동 & assets.csv 자동 업데이트

이 Notebook에서 하는 일:
1. assets.csv에서 ticker(종목코드)가 있는 자산 추출
2. FinanceDataReader / pykrx로 현재 시세 조회
3. current_value 자동 계산 (수량 × 현재가)
4. 현금성 자산(ticker 없음)은 수동 입력 유지
5. 업데이트된 assets.csv 저장

> **실행 순서**: 이 노트북 실행 후 → 01_data_input.ipynb 실행하면 최신 시세로 분석됩니다.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date, timedelta
import time

PROJECT_ROOT = Path.cwd()
assets_path  = PROJECT_ROOT / 'assets.csv'
sample_path  = PROJECT_ROOT / 'assets_sample.csv'

# assets.csv 로드
if assets_path.exists():
    df = pd.read_csv(assets_path, encoding='utf-8-sig', dtype={'ticker': str})
    print(f'assets.csv 로드 — {len(df)}개 자산')
else:
    df = pd.read_csv(sample_path, encoding='utf-8-sig', dtype={'ticker': str})
    print(f'샘플 데이터 로드 — {len(df)}개 자산')

# ticker 정리 — float으로 읽힌 경우(예: '153130.0') 정수 문자열로 변환
def clean_ticker(x):
    s = str(x).strip()
    if s in ('', 'nan', 'NaN', 'None'):
        return ''
    try:
        return str(int(float(s)))
    except:
        return s

df['ticker']        = df['ticker'].apply(clean_ticker)
df['current_value'] = pd.to_numeric(df['current_value'], errors='coerce').fillna(0)
df['quantity']      = pd.to_numeric(df['quantity'],      errors='coerce').fillna(0)
df['unit_price']    = pd.to_numeric(df['unit_price'],    errors='coerce').fillna(0)

# ticker 있는 것 / 없는 것 분리
has_ticker = df['ticker'] != ''
print(f'\n시세 조회 대상: {has_ticker.sum()}개')
print(f'수동 유지 대상: {(~has_ticker).sum()}개 (현금성 자산)')
print()
print(df[has_ticker][['asset_name','ticker','quantity','current_value']].to_string(index=False))

샘플 데이터 로드 — 7개 자산

시세 조회 대상: 5개
수동 유지 대상: 2개 (현금성 자산)

    asset_name ticker  quantity  current_value
TIGER 미국S&P500 360750      50.0        1020000
    KODEX 단기채권 153130     100.0       10250000
 TDF2035(미래에셋) 189400      30.0         480000
   KODEX 국고채3년 114820     200.0       20400000
TIGER 리츠부동산인프라 329200      80.0         400000


## 1. 시세 조회 함수

In [2]:
import FinanceDataReader as fdr

def get_price_fdr(ticker):
    try:
        ticker = ticker.zfill(6)
        end    = date.today()
        start  = end - timedelta(days=10)
        df_p   = fdr.DataReader(ticker, start=str(start), end=str(end))
        if df_p.empty:
            return None, None
        return float(df_p['Close'].iloc[-1]), str(df_p.index[-1].date())
    except:
        return None, None

def get_price_pykrx(ticker):
    try:
        from pykrx import stock
        ticker = ticker.zfill(6)
        today  = date.today().strftime('%Y%m%d')
        start  = (date.today() - timedelta(days=10)).strftime('%Y%m%d')
        df_p   = stock.get_market_ohlcv_by_date(start, today, ticker)
        if df_p.empty:
            return None, None
        return float(df_p['종가'].iloc[-1]), str(df_p.index[-1].date())
    except:
        return None, None

def get_price(ticker):
    price, pdate = get_price_fdr(ticker)
    if price:
        return price, pdate, 'FDR'
    price, pdate = get_price_pykrx(ticker)
    if price:
        return price, pdate, 'pykrx'
    return None, None, 'failed'

print('시세 조회 함수 준비 완료')

시세 조회 함수 준비 완료


## 2. 실시간 시세 조회

In [3]:
print('=== 실시간 시세 조회 중 ===')
print()

df_updated = df.copy()
results    = []

for idx, row in df[has_ticker].iterrows():
    ticker     = row['ticker']
    asset_name = row['asset_name']
    quantity   = row['quantity']
    old_value  = row['current_value']

    price, price_date, source = get_price(ticker)
    time.sleep(0.3)

    if price and quantity > 0:
        new_value  = price * quantity
        change     = new_value - old_value
        change_pct = (change / old_value * 100) if old_value > 0 else 0
        df_updated.at[idx, 'unit_price']    = price
        df_updated.at[idx, 'current_value'] = new_value
        print(f'✅ [{ticker}] {asset_name}')
        print(f'   현재가: {price:>10,.0f}원  수량: {quantity:.0f}주')
        print(f'   평가액: {new_value:>12,.0f}원  (변동: {change:+,.0f}원 / {change_pct:+.1f}%)')
        print(f'   기준일: {price_date}  출처: {source}')
        results.append({'status': 'ok', 'name': asset_name, 'change': change})
    elif price and quantity == 0:
        print(f'⚠️  [{ticker}] {asset_name} — 수량 0, current_value 수동 유지')
        results.append({'status': 'manual', 'name': asset_name, 'change': 0})
    else:
        print(f'❌  [{ticker}] {asset_name} — 시세 조회 실패, 기존값 유지')
        results.append({'status': 'failed', 'name': asset_name, 'change': 0})
    print()

ok   = sum(1 for r in results if r['status'] == 'ok')
fail = sum(1 for r in results if r['status'] == 'failed')
print(f'조회 완료: {ok}개 성공 / {fail}개 실패')

=== 실시간 시세 조회 중 ===

✅ [360750] TIGER 미국S&P500
   현재가:     27,475원  수량: 50주
   평가액:    1,373,750원  (변동: +353,750원 / +34.7%)
   기준일: 2026-05-20  출처: FDR

✅ [153130] KODEX 단기채권
   현재가:    113,205원  수량: 100주
   평가액:   11,320,500원  (변동: +1,070,500원 / +10.4%)
   기준일: 2026-05-20  출처: FDR

✅ [189400] TDF2035(미래에셋)
   현재가:     24,720원  수량: 30주
   평가액:      741,600원  (변동: +261,600원 / +54.5%)
   기준일: 2026-05-20  출처: FDR

✅ [114820] KODEX 국고채3년
   현재가:    104,160원  수량: 200주
   평가액:   20,832,000원  (변동: +432,000원 / +2.1%)
   기준일: 2026-05-20  출처: FDR

✅ [329200] TIGER 리츠부동산인프라
   현재가:      4,195원  수량: 80주
   평가액:      335,600원  (변동: -64,400원 / -16.1%)
   기준일: 2026-05-20  출처: FDR

조회 완료: 5개 성공 / 0개 실패


## 3. 현금성 자산 수동 업데이트 (선택)

In [4]:
cash_df = df[~has_ticker][['account_name','asset_name','asset_type','current_value']]
print('=== 현금성 자산 (수동 업데이트) ===')
for idx, row in cash_df.iterrows():
    print(f'  [{idx}] {row["account_name"]} — {row["asset_name"]}: {row["current_value"]:,.0f}원')
print()
print('금액 변경 시 아래 딕셔너리에 index번호: 새금액 형식으로 입력 후 다음 셀 실행')

=== 현금성 자산 (수동 업데이트) ===
  [4] 예금(예시) — 정기예금(KB): 30,000,000원
  [5] 예금(예시) — CMA(삼성): 5,000,000원

금액 변경 시 아래 딕셔너리에 index번호: 새금액 형식으로 입력 후 다음 셀 실행


In [5]:
# 변경할 현금성 자산 입력 (변동 없으면 빈 딕셔너리 그대로 실행)
# 예시: cash_updates = { 4: 32000000, 5: 6000000 }
cash_updates = {}

for idx, new_val in cash_updates.items():
    old_val = df_updated.at[idx, 'current_value']
    df_updated.at[idx, 'current_value'] = new_val
    print(f'업데이트: [{idx}] {df_updated.at[idx, "asset_name"]} {old_val:,.0f} → {new_val:,.0f}원')

if not cash_updates:
    print('현금성 자산 변동 없음 — 기존값 유지')

현금성 자산 변동 없음 — 기존값 유지


## 4. 변경 확인 및 저장

In [6]:
import shutil

old_total = df['current_value'].sum()
new_total = df_updated['current_value'].sum()
diff      = new_total - old_total

print('=== 업데이트 전후 비교 ===')
print(f'업데이트 전: {old_total:>15,.0f}원')
print(f'업데이트 후: {new_total:>15,.0f}원')
print(f'변동액:      {diff:>+15,.0f}원  ({diff/old_total*100:+.2f}%)')
print()

# 백업 후 저장
today_str  = date.today().strftime('%Y%m%d')
backup_dir = PROJECT_ROOT / 'backup'
backup_dir.mkdir(exist_ok=True)

if assets_path.exists():
    shutil.copy(assets_path, backup_dir / f'assets_{today_str}.csv')
    print(f'백업 저장: backup/assets_{today_str}.csv')

df_updated.to_csv(assets_path, index=False, encoding='utf-8-sig')
print(f'assets.csv 업데이트 완료!')
print()
print('다음: 01_data_input.ipynb 실행하여 최신 시세로 분석하세요.')

=== 업데이트 전후 비교 ===
업데이트 전:      67,550,000원
업데이트 후:      69,603,450원
변동액:           +2,053,450원  (+3.04%)

assets.csv 업데이트 완료!

다음: 01_data_input.ipynb 실행하여 최신 시세로 분석하세요.


---
## 완료!
- ticker 있는 자산: 실시간 시세 자동 업데이트
- 현금성 자산: 수동 입력 유지
- 기존 파일: `backup/` 폴더에 날짜별 자동 보관

**다음 단계**: `01_data_input.ipynb` → `02~05` → `07_report.ipynb` 순서로 실행